### Justifying our Attribute Selection:
1. **RestaurantsPriceRange2:** Budget is one of the largest non-negotiables when deciding on restaurants for individuals.
2. **OutdoorSeating:** Given the nice climate of Santa Barbara, the availability of outdoor dining would probably influence user choices.
3. **RestaurantsTakeOut:** Takeout has only become a larger trend in recent years and therefore, the availability of it might be a large determinant.

In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from shared import load_data, evaluate_model

In [3]:
train_df, test_df, restaurants_df = load_data()

def parse_attributes(attr_str):
    if pd.isna(attr_str):
        return {}
    try:
        return ast.literal_eval(attr_str)
    except:
        return {}

restaurants_df['parsed_attributes'] = restaurants_df['attributes'].apply(parse_attributes)

restaurants_df['Price'] = restaurants_df['parsed_attributes'].apply(lambda x: str(x.get('RestaurantsPriceRange2', 'Unknown')))
restaurants_df['Outdoor'] = restaurants_df['parsed_attributes'].apply(lambda x: str(x.get('OutdoorSeating', 'Unknown')))
restaurants_df['TakeOut'] = restaurants_df['parsed_attributes'].apply(lambda x: str(x.get('RestaurantsTakeOut', 'Unknown')))

In [4]:
def create_feature_string(row):
    cats = str(row['categories']).replace(',', ' ')
    price = f"Price_{row['Price']} "
    outdoor = f"Outdoor_{row['Outdoor']} "
    takeout = f"Takeout_{row['TakeOut']} "
    return cats + " " + price + outdoor + takeout

restaurants_df['content_features'] = restaurants_df.apply(create_feature_string, axis=1)

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(restaurants_df['content_features'])

item_indices = pd.Series(restaurants_df.index, index=restaurants_df['business_id']).to_dict()

In [5]:
positive_train = train_df[train_df['stars'] >= 4.0]

user_profiles = {}
for user, group in positive_train.groupby('user_id'):
    liked_item_ids = group['business_id'].tolist()
    liked_indices = [item_indices[biz] for biz in liked_item_ids if biz in item_indices]

    if liked_indices:
        user_profile_vector = np.asarray(tfidf_matrix[liked_indices].mean(axis=0))
        user_profiles[user] = user_profile_vector

In [7]:
test_users = test_df['user_id'].unique()
all_businesses = restaurants_df['business_id'].tolist()
predictions = {}

for user in test_users:
    if user in user_profiles:
        sim_scores = cosine_similarity(user_profiles[user], tfidf_matrix).flatten()

        top_indices = sim_scores.argsort()[-10:][::-1]
        predictions[user] = [all_businesses[i] for i in top_indices]
    else:
        predictions[user] = []

metrics = evaluate_model(predictions, test_df, k=10)
print("Content-Based (A) Performance:")
print("-" * 30)
print(f"Hit Rate @ 10: {metrics['Hit@10']:.4f}")
print(f"NDCG @ 10:     {metrics['NDCG@10']:.4f}")

Content-Based (A) Performance:
------------------------------
Hit Rate @ 10: 0.0215
NDCG @ 10:     0.0086


In [8]:
print("--- User Scenario Demo ---")
print("Persona: Focused on high-protein, lean meals. Prefers takeout for meal prep convenience, moderate price.\n")

# 1. Define the ideal features for this specific scenario
demo_features = "salad seafood health food Takeout_True Price_2"

# 2. Vectorize this theoretical user
demo_vector = tfidf.transform([demo_features])

# 3. Calculate similarity against all Santa Barbara restaurants
demo_sim_scores = cosine_similarity(demo_vector, tfidf_matrix).flatten()

# 4. Get the Top 5 recommendations
top_5_demo_indices = demo_sim_scores.argsort()[-5:][::-1]

print("Top 5 Recommendations:")
for rank, idx in enumerate(top_5_demo_indices, 1):
    row = restaurants_df.iloc[idx]
    print(f"{rank}. {row['name']}")
    print(f"   Categories: {row['categories']}")
    print(f"   Attributes: Price={row['Price']}, TakeOut={row['TakeOut']}, Outdoor={row['Outdoor']}\n")

--- User Scenario Demo ---
Persona: Focused on high-protein, lean meals. Prefers takeout for meal prep convenience, moderate price.

Top 5 Recommendations:
1. Chuck's of Hawaii
   Categories: salad, steakhouses, restaurants, nightlife, seafood
   Attributes: Price=2, TakeOut=True, Outdoor=True

2. Santa Barbara Bay Cafe
   Categories: seafood, restaurants
   Attributes: Price=2, TakeOut=True, Outdoor=True

3. Corazon Cocina
   Categories: restaurants, seafood, mexican
   Attributes: Price=2, TakeOut=True, Outdoor=True

4. Casa Blanca Restaurant
   Categories: mexican, restaurants, seafood
   Attributes: Price=2, TakeOut=True, Outdoor=True

5. Isabella Gourmet Foods
   Categories: restaurants, food, specialty food, farmers market, grocery, health markets
   Attributes: Price=2, TakeOut=True, Outdoor=Unknown

